In [103]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("Spark SQL Practice")
    .config("spark.master", "local[*]")
    .config("spark.sql.warehouse.dir", "src/main/resources/warehouse") # this will specify where to store our tables
    .config("spark.sql.legacy.allowCreatingMangedTableUsingNonemptyLocation", "true")
    .getOrCreate()
)


26/03/22 14:59:02 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [104]:
carsDF = spark.read.json("src/main/resources/data/cars.json")

carsDF.show(5)

+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|Acceleration|Cylinders|Displacement|Horsepower|Miles_per_Gallon|                Name|Origin|Weight_in_lbs|      Year|
+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|        12.0|        8|       307.0|       130|            18.0|chevrolet chevell...|   USA|         3504|1970-01-01|
|        11.5|        8|       350.0|       165|            15.0|   buick skylark 320|   USA|         3693|1970-01-01|
|        11.0|        8|       318.0|       150|            18.0|  plymouth satellite|   USA|         3436|1970-01-01|
|        12.0|        8|       304.0|       150|            16.0|       amc rebel sst|   USA|         3433|1970-01-01|
|        10.5|        8|       302.0|       140|            17.0|         ford torino|   USA|         3449|1970-01-01|
+------------+---------+------------+----------+

In [105]:
# spark sql

# this creates an alias in spark so i can refer to this dataframe as a table
carsDF.createOrReplaceTempView("cars")


# whenever u run sql u return a dataframe
americanCarsDF = spark.sql(
    """
        select Name from cars
        where Origin = 'USA'
    """
)

americanCarsDF.show(5)

+--------------------+
|                Name|
+--------------------+
|chevrolet chevell...|
|   buick skylark 320|
|  plymouth satellite|
|       amc rebel sst|
|         ford torino|
+--------------------+
only showing top 5 rows



In [106]:
# this returns an empty dataframe, and creates a new folder in our project 
# src\main\resources\warehouse\rtjvm.db 
# this folder will store all our databases!!
spark.sql("DROP DATABASE IF EXISTS rtjvm CASCADE")
spark.sql("CREATE DATABASE rtjvm")

DataFrame[]

In [107]:
# now every subsequent select will be related to the rtjvm database
spark.sql("use rtjvm")

DataFrame[]

In [108]:
# see all databases
databasesDF = spark.sql("show databases")

databasesDF.show()

+---------+
|namespace|
+---------+
|  default|
|    rtjvm|
+---------+



In [109]:
# transfer tables from a database to a spark table
employeesDB = (
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("user", "docker")
    .option("password", "docker")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .option("dbtable", "public.employees")
    .load()
)

# this will SAVE employees as a table into the current database we are using from the `use rtjvm` command
# after this runs we should see an employees folder within our warehouse
# src\main\resources\warehouse\rtjvm.db
(
    employeesDB.write
    .mode("Overwrite")
    .saveAsTable("employees")
)

In [113]:
def readTable(tableName):
   return (
    spark.read
    .format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("user", "docker")
    .option("password", "docker")
    .option("url", "jdbc:postgresql://postgres:5432/rtjvm")
    .option("dbtable", f"public.{tableName}")
    .load())


def transferTables(tableNames):
   for tableName in tableNames:
      tableDF = readTable(tableName)
      # load it into memory so we can refer to it in spark sql
      tableDF.createOrReplaceGlobalTempView(tableName)
      # save as table
      tableDF.write.mode("Overwrite").saveAsTable(tableName)
 

In [114]:
# transfer all tables to our warehouse - so from regular database to spark table (datawarehouse)
transferTables(["employees", "departments", "titles", "dept_emp", "salaries", "dept_manager"])

In [ ]:
# read df from loaded tables
# essentially convert a table loaded into spark into a dataframe
employeesDF2 = spark.read.table("employees")

In [116]:
# Exercises
# 1. read the movies DF and store it as a spark table in rtjvm db
# 2. count how many employees were hired between jan 1 1999 and jan 1 2000
# 3. show the average salaries for the employees hired in between those dates grouped by department
# 4. show the name of the best-paying department for employees hired in between those dates

In [117]:
# exercise 1 
moviesDF = spark.read.json("src/main/resources/data/movies.json")

moviesDF.write.mode("Overwrite").saveAsTable("movies")


describedDF = spark.sql("DESCRIBE EXTENDED movies")

describedDF.show(50)

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|       Creative_Type|              string|   NULL|
|            Director|              string|   NULL|
|         Distributor|              string|   NULL|
|         IMDB_Rating|              double|   NULL|
|          IMDB_Votes|              bigint|   NULL|
|         MPAA_Rating|              string|   NULL|
|         Major_Genre|              string|   NULL|
|   Production_Budget|              bigint|   NULL|
|        Release_Date|              string|   NULL|
|Rotten_Tomatoes_R...|              bigint|   NULL|
|    Running_Time_min|              bigint|   NULL|
|              Source|              string|   NULL|
|               Title|              string|   NULL|
|        US_DVD_Sales|              bigint|   NULL|
|            US_Gross|              bigint|   NULL|
|     Worldwide_Gross|              bigint|   NULL|
|           

In [138]:
# exercise 2
spark.sql("DESCRIBE employees").show()

spark.sql("SELECT * FROM employees LIMIT 5").show()

employeesSubset = spark.sql(
     """
     SELECT * FROM employees
     WHERE hire_date > '1999-01-01' and hire_date < '2000-01-01'
     """
)

employeesSubset.createOrReplaceTempView("employeesSubset")

spark.sql("SELECT COUNT(*) FROM employeesSubset").show()

+----------+-------------------+-------+
|  col_name|          data_type|comment|
+----------+-------------------+-------+
|    emp_no|                int|   NULL|
|birth_date|               date|   NULL|
|first_name|        varchar(14)|   NULL|
| last_name|        varchar(16)|   NULL|
|    gender|varchar(2147483647)|   NULL|
| hire_date|               date|   NULL|
+----------+-------------------+-------+

+------+----------+----------+---------+------+----------+
|emp_no|birth_date|first_name|last_name|gender| hire_date|
+------+----------+----------+---------+------+----------+
| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24|
| 10020|1952-12-24|    Mayuko|  Warwick|     M|1991-01-26|
| 10030|1958-07-14|     Elvis|  Demeyer|     M|1994-02-17|
| 10040|1959-09-13|     Weiyi|  Meriste|     F|1993-02-14|
| 10050|1958-05-21|   Yinghua|   Dredge|     M|1990-12-25|
+------+----------+----------+---------+------+----------+

+--------+
|count(1)|
+--------+
|     152|
+--------+



In [119]:
# Exercise 3.1 analyze data

# read salaries
spark.sql("DESCRIBE salaries").show()

spark.sql("SELECT * FROM salaries LIMIT 5").show()

+---------+---------+-------+
| col_name|data_type|comment|
+---------+---------+-------+
|   emp_no|      int|   NULL|
|   salary|      int|   NULL|
|from_date|     date|   NULL|
|  to_date|     date|   NULL|
+---------+---------+-------+

+------+------+----------+----------+
|emp_no|salary| from_date|   to_date|
+------+------+----------+----------+
| 10010| 72488|1996-11-24|1997-11-24|
| 10010| 74347|1997-11-24|1998-11-24|
| 10010| 75405|1998-11-24|1999-11-24|
| 10010| 78194|1999-11-24|2000-11-23|
| 10010| 79580|2000-11-23|2001-11-23|
+------+------+----------+----------+



In [122]:
# Exercise 3.2 join salaries and employees

employeesWithSalaries = spark.sql(
    """
    SELECT * FROM employees e
    JOIN salaries s USING (emp_no) 
    """
)

employeesWithSalaries.createOrReplaceTempView("employeesWithSalaries")


spark.sql(
    """
    SELECT * FROM employeesWithSalaries LIMIT 5
    """).show()

+------+----------+----------+---------+------+----------+------+----------+----------+
|emp_no|birth_date|first_name|last_name|gender| hire_date|salary| from_date|   to_date|
+------+----------+----------+---------+------+----------+------+----------+----------+
| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24| 72488|1996-11-24|1997-11-24|
| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24| 74347|1997-11-24|1998-11-24|
| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24| 75405|1998-11-24|1999-11-24|
| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24| 78194|1999-11-24|2000-11-23|
| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24| 79580|2000-11-23|2001-11-23|
+------+----------+----------+---------+------+----------+------+----------+----------+



In [129]:
# Exercise 3.3 join with deparment

employeesWithDept = spark.sql(
    """
    SELECT e.*, d.dept_name FROM departments d
    JOIN (
        SELECT * FROM employeesWithSalaries e
        JOIN dept_emp d ON e.emp_no = d.emp_no 
    ) e USING (dept_no)
    """
)

employeesWithDept.createOrReplaceTempView("employeesWithDept")
employeesWithDept.show(5)


+-------+------+----------+----------+---------+------+----------+------+----------+----------+------+----------+----------+------------------+
|dept_no|emp_no|birth_date|first_name|last_name|gender| hire_date|salary| from_date|   to_date|emp_no| from_date|   to_date|         dept_name|
+-------+------+----------+----------+---------+------+----------+------+----------+----------+------+----------+----------+------------------+
|   d006| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24| 72488|1996-11-24|1997-11-24| 10010|2000-06-26|9999-01-01|Quality Management|
|   d004| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24| 72488|1996-11-24|1997-11-24| 10010|1996-11-24|2000-06-26|        Production|
|   d006| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24| 74347|1997-11-24|1998-11-24| 10010|2000-06-26|9999-01-01|Quality Management|
|   d004| 10010|1963-06-01| Duangkaew| Piveteau|     F|1989-08-24| 74347|1997-11-24|1998-11-24| 10010|1996-11-24|2000-06-26|        Prod

In [134]:
# 3. show salaries grouped by department
averageSalariesByDept = spark.sql(
    """
    SELECT dept_no, dept_name, AVG(salary) as average_salary FROM employeesWithDept
    GROUP BY dept_no, dept_name
    """
)

averageSalariesByDept.createOrReplaceTempView("averageSalariesByDept")
averageSalariesByDept.show(5)

+-------+---------------+------------------+
|dept_no|      dept_name|    average_salary|
+-------+---------------+------------------+
|   d007|          Sales| 80457.91612429355|
|   d004|     Production| 59543.16866035571|
|   d002|        Finance| 70678.28583992964|
|   d003|Human Resources|55782.081478444234|
|   d001|      Marketing| 71796.87627136416|
+-------+---------------+------------------+
only showing top 5 rows



In [137]:
# show name of best-paying departments

spark.sql("SELECT * FROM averageSalariesByDept ORDER BY average_salary DESC").show(10)

+-------+------------------+------------------+
|dept_no|         dept_name|    average_salary|
+-------+------------------+------------------+
|   d007|             Sales| 80457.91612429355|
|   d001|         Marketing| 71796.87627136416|
|   d002|           Finance| 70678.28583992964|
|   d008|          Research| 59705.72610183135|
|   d005|       Development| 59688.53897495259|
|   d004|        Production| 59543.16866035571|
|   d009|  Customer Service| 58526.17977424557|
|   d006|Quality Management| 56956.49025613943|
|   d003|   Human Resources|55782.081478444234|
+-------+------------------+------------------+

